In [ ]:
import sys
import platform
import pkg_resources
import pandas as pd

# ✅ Define key libraries for your pipeline
important_libs = [
    "pandas", "numpy", "scikit-learn", "lightgbm",
    "torch", "transformers", "optuna", "flask"
]

def check_versions(env_name="local"):
    print(f"\n🔍 Environment Check: {env_name.upper()}")
    print("-" * 50)
    print(f"Python version: {platform.python_version()}")
    print(f"System: {platform.system()} {platform.release()} ({platform.machine()})\n")

    data = []
    for lib in important_libs:
        try:
            version = pkg_resources.get_distribution(lib).version
            data.append([lib, version])
        except Exception:
            data.append([lib, "❌ Not Installed"])

    df = pd.DataFrame(data, columns=["Library", "Version"])
    print(df.to_string(index=False))
    return df

if __name__ == "__main__":
    df_versions = check_versions("colab_or_local")

    # Save to file for easy comparison
    df_versions.to_csv("library_versions.csv", index=False)
    print("\n💾 Versions saved to library_versions.csv")


/tmp/ipython-input-4036432475.py:3: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources



🔍 Environment Check: COLAB_OR_LOCAL
--------------------------------------------------
Python version: 3.12.12
System: Linux 6.6.105+ (x86_64)

     Library         Version
      pandas           2.2.2
       numpy           2.0.2
scikit-learn           1.6.1
    lightgbm           4.6.0
       torch     2.8.0+cu126
transformers          4.57.1
      optuna ❌ Not Installed
       flask           3.1.2

💾 Versions saved to library_versions.csv


In [ ]:
!pip install numpy==1.26.4 optuna==4.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 35.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but y

In [ ]:
# =========================================================
#  FinBERT Sentiment Extraction for Stock News
# =========================================================
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# ---------------------------------------------------------
# 1. Load Cleaned Dataset
# ---------------------------------------------------------
input_path = "/content/cleaned_stocknews.csv"   # update if needed
df = pd.read_csv(input_path)
print(f"✅ Loaded {len(df)} rows from {input_path}")

# ---------------------------------------------------------
# 2. Load FinBERT Model and Tokenizer
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🧠 Using device:", device)

model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

# ---------------------------------------------------------
# 3. Define Sentiment Scoring Function
# ---------------------------------------------------------
def get_finbert_sentiment(text):
    """
    Returns sentiment score = Positive - Negative
    """
    try:
        inputs = tokenizer(
            text, return_tensors="pt",
            truncation=True, padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            # order: [negative, neutral, positive]
            score = probs[0][2].item() - probs[0][0].item()
        return score
    except Exception as e:
        print("⚠️ Error processing text:", e)
        return 0.0

# ---------------------------------------------------------
# 4. Apply FinBERT to All News
# ---------------------------------------------------------
tqdm.pandas()
df["sentiment_score"] = df["Clean_News"].progress_apply(get_finbert_sentiment)

# ---------------------------------------------------------
# 5. Save Result
# ---------------------------------------------------------
output_path = "/content/finbert_sentiment.csv"
df.to_csv(output_path, index=False)
print(f"💾 Saved FinBERT sentiment file: {output_path}")

# Quick preview
print(df.head())


✅ Loaded 1989 rows from /content/cleaned_stocknews.csv
🧠 Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

  0%|          | 0/1989 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

100%|██████████| 1989/1989 [01:04<00:00, 30.80it/s]


💾 Saved FinBERT sentiment file: /content/finbert_sentiment.csv
         Date  Label                                         Clean_News  \
0  2008-08-08      0  bgeorgia downs two russian warplanes as countr...   
1  2008-08-11      1  bwhy wont america and nato help us if they won...   
2  2008-08-12      0  bremember that adorable yearold who sang at th...   
3  2008-08-13      0  b us refuses israel weapons to attack iran rep...   
4  2008-08-14      1  ball the experts admit that we should legalise...   

   sentiment_score  
0        -0.993480  
1        -0.998100  
2        -0.973749  
3        -0.770274  
4        -0.999579  


In [ ]:
# =========================================================
#  LightGBM Training using FinBERT Sentiment
# =========================================================
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import optuna

# ---------------------------------------------------------
# 1. Load Dataset
# ---------------------------------------------------------
data_path = "/content/finbert_sentiment.csv"
df = pd.read_csv(data_path)
print(f"✅ Loaded {df.shape[0]} rows for training")

# ---------------------------------------------------------
# 2. Feature Engineering
# ---------------------------------------------------------
df["sentiment_norm"] = (df["sentiment_score"] - df["sentiment_score"].min()) / (
    df["sentiment_score"].max() - df["sentiment_score"].min()
)
X = df[["sentiment_score", "sentiment_norm"]]
y = df["Label"]

# ---------------------------------------------------------
# 3. Split Dataset
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)
print(f"📊 Train size: {len(X_train)} | Test size: {len(X_test)}")

# ---------------------------------------------------------
# 4. Hyperparameter Tuning (Optuna)
# ---------------------------------------------------------
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "binary_error",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "num_leaves": trial.suggest_int("num_leaves", 15, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 50),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds)

print("⚙️ Running Optuna optimization... (this takes ~1 minute)")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
best_params = study.best_params
print("🏆 Best parameters:", best_params)

# ---------------------------------------------------------
# 5. Train Final LightGBM Model
# ---------------------------------------------------------
model = lgb.LGBMClassifier(**best_params)
model.fit(X_train, y_train)
preds = model.predict(X_test)

acc = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds)
print(f"✅ Accuracy: {acc:.4f} | F1 Score: {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_test, preds))

# ---------------------------------------------------------
# 6. Save Model
# ---------------------------------------------------------
model.booster_.save_model("/content/lightgbm_model.txt")
joblib.dump(model, "/content/lightgbm_model.pkl")
print("💾 Model saved as lightgbm_model.txt and lightgbm_model.pkl")


[I 2025-10-19 16:29:42,973] A new study created in memory with name: no-name-5f0dd76d-3a5b-4d2a-9070-1b864a3b12de


✅ Loaded 1989 rows for training
📊 Train size: 1591 | Test size: 398
⚙️ Running Optuna optimization... (this takes ~1 minute)


[I 2025-10-19 16:29:43,494] Trial 0 finished with value: 0.496551724137931 and parameters: {'num_leaves': 114, 'learning_rate': 0.041311887436508724, 'feature_fraction': 0.9893499616464935, 'bagging_fraction': 0.9883560827011649, 'bagging_freq': 5, 'min_data_in_leaf': 10}. Best is trial 0 with value: 0.496551724137931.
[I 2025-10-19 16:29:43,646] Trial 1 finished with value: 0.517162471395881 and parameters: {'num_leaves': 130, 'learning_rate': 0.08623987022668714, 'feature_fraction': 0.8093826779975981, 'bagging_fraction': 0.6871597220478284, 'bagging_freq': 7, 'min_data_in_leaf': 20}. Best is trial 1 with value: 0.517162471395881.
[I 2025-10-19 16:29:43,783] Trial 2 finished with value: 0.5248868778280543 and parameters: {'num_leaves': 94, 'learning_rate': 0.29022118244997097, 'feature_fraction': 0.8013009525747914, 'bagging_fraction': 0.7360140351662674, 'bagging_freq': 8, 'min_data_in_leaf': 42}. Best is trial 2 with value: 0.5248868778280543.
[I 2025-10-19 16:29:43,946] Trial 3 fi

🏆 Best parameters: {'num_leaves': 140, 'learning_rate': 0.0016770417303829971, 'feature_fraction': 0.5874828596762883, 'bagging_fraction': 0.9879710755040217, 'bagging_freq': 6, 'min_data_in_leaf': 50}
✅ Accuracy: 0.5075 | F1 Score: 0.6733

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       196
           1       0.51      1.00      0.67       202

    accuracy                           0.51       398
   macro avg       0.25      0.50      0.34       398
weighted avg       0.26      0.51      0.34       398

💾 Model saved as lightgbm_model.txt and lightgbm_model.pkl


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip install  lightgbm
